In [1]:
# ============================================================
# PRODUCT PRICING & BUNDLING ANALYSIS
# FINAL BUSINESS INSIGHTS
# ============================================================

import pandas as pd
import numpy as np


In [2]:
# ============================================================
# 1. LOAD FINAL OUTPUTS
# ============================================================

sales_df = pd.read_csv("pricing_sales_clean.csv")

product_elasticity = pd.read_csv(
    "product_elasticity.csv"
)

pricing_opportunities = pd.read_csv(
    "final_pricing_opportunities.csv"
)

bundle_recommendations = pd.read_csv(
    "final_bundle_recommendations.csv"
)

data_quality_summary = pd.read_csv(
    "data_quality_summary.csv"
)


C:\Users\Dell\AppData\Local\Temp\ipykernel_7044\700010830.py:5: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  sales_df = pd.read_csv("pricing_sales_clean.csv")


In [3]:
# ============================================================
# 2. OVERALL BUSINESS KPIs
# ============================================================

total_revenue = sales_df["Revenue"].sum()

total_units = sales_df["Quantity"].sum()

total_orders = sales_df["Invoice_ID"].nunique()

total_products = sales_df["Stock_Code"].nunique()

total_customers = sales_df["Customer_ID"].nunique()

average_order_value = total_revenue / total_orders

average_unit_price = sales_df["Unit_Price"].mean()


print("========== FINAL BUSINESS KPIs ==========")

print(f"Total Revenue : £{total_revenue:,.2f}")

print(f"Total Units Sold : {total_units:,.0f}")

print(f"Total Orders : {total_orders:,}")

print(f"Unique Products : {total_products:,}")

print(f"Unique Customers : {total_customers:,}")

print(f"Average Order Value : £{average_order_value:,.2f}")

print(f"Average Unit Price : £{average_unit_price:,.2f}")

========== FINAL BUSINESS KPIs ==========
Total Revenue : £10,272,136.23
Total Units Sold : 5,812,948
Total Orders : 20,953
Unique Products : 4,250
Unique Customers : 4,312
Average Order Value : £490.25
Average Unit Price : £4.27


In [4]:
# ============================================================
# 3. TOP REVENUE PRODUCTS
# ============================================================

top_products = (
    sales_df
    .groupby("Stock_Code")
    .agg(
        Revenue=("Revenue", "sum"),
        Quantity=("Quantity", "sum"),
        Orders=("Invoice_ID", "nunique")
    )
    .reset_index()
    .sort_values(
        "Revenue",
        ascending=False
    )
)

print("\n========== TOP REVENUE PRODUCTS ==========")

display(
    top_products.head(10)
)


========== TOP REVENUE PRODUCTS ==========


,Stock_Code,Revenue,Quantity,Orders
4235,M,262950.93,2765,513
1634,22423,169912.76,13685,2019
3640,85123A,158305.72,58386,3281
4234,DOT,116408.71,730,730
3612,85099B,88976.33,49804,1966
3336,84879,72890.19,45228,1412
1322,22086,58127.30,17205,957
2508,47566,49664.12,10079,1013
4237,POST,49477.54,2310,759
2945,84347,47954.49,23037,347


In [5]:
# ============================================================
# 4. PRICE BAND PERFORMANCE
# ============================================================

price_band_analysis = (
    sales_df
    .groupby("Price_Band", observed=True)
    .agg(
        Revenue=("Revenue", "sum"),
        Units_Sold=("Quantity", "sum"),
        Orders=("Invoice_ID", "nunique"),
        Products=("Stock_Code", "nunique")
    )
    .reset_index()
)

price_band_analysis["Revenue_Share_%"] = (
    price_band_analysis["Revenue"]
    / total_revenue
    * 100
)

price_band_analysis = price_band_analysis.sort_values(
    "Revenue",
    ascending=False
)

print("\n========== PRICE BAND PERFORMANCE ==========")

display(
    price_band_analysis
)




========== PRICE BAND PERFORMANCE ==========


,Price_Band,Revenue,Units_Sold,Orders,Products,Revenue_Share_%
2,2-5,3397038.670,1098400,17609,2522,33.070421
0,1-2,2675816.070,1814212,17023,1855,26.049266
4,5-10,1684403.740,239625,13774,1521,16.397794
6,<=1,1204722.944,2600552,14465,1258,11.728066
1,10-25,706311.940,53860,7557,698,6.875999
5,50+,444942.830,1133,894,48,4.331551
3,25-50,158900.040,5166,1312,161,1.546904


In [6]:
# ============================================================
# 5. ELASTICITY SUMMARY
# ============================================================

elasticity_summary = (
    product_elasticity
    .groupby("Elasticity_Type")
    .agg(
        Products=("Stock_Code", "nunique"),
        Total_Revenue=("Total_Revenue", "sum"),
        Total_Quantity=("Total_Quantity", "sum")
    )
    .reset_index()
)

elasticity_summary["Revenue_Share_%"] = (
    elasticity_summary["Total_Revenue"]
    / elasticity_summary["Total_Revenue"].sum()
    * 100
)

print("\n========== ELASTICITY SUMMARY ==========")

display(
    elasticity_summary
)


========== ELASTICITY SUMMARY ==========


,Elasticity_Type,Products,Total_Revenue,Total_Quantity,Revenue_Share_%
0,Elastic,1296,1612423.47,948306,35.428391
1,Inelastic,615,1143472.50,569522,25.124535
2,Positive / Unusual,1371,1795322.52,988927,39.447074


In [7]:
# ============================================================
# 6. HIGH-REVENUE ELASTIC PRODUCTS
# ============================================================

high_revenue_elastic = product_elasticity[
    product_elasticity["Elasticity_Type"] == "Elastic"
].copy()

high_revenue_elastic = high_revenue_elastic.sort_values(
    "Total_Revenue",
    ascending=False
)

print("\n========== HIGH-REVENUE ELASTIC PRODUCTS ==========")

display(
    high_revenue_elastic[
        [
            "Stock_Code",
            "Average_Elasticity",
            "Average_Price",
            "Total_Quantity",
            "Total_Revenue",
            "Number_of_Months",
            "Elasticity_Type"
        ]
    ].head(10)
)


========== HIGH-REVENUE ELASTIC PRODUCTS ==========


,Stock_Code,Average_Elasticity,Average_Price,Total_Quantity,Total_Revenue,Number_of_Months,Elasticity_Type
894,85099B,-1.077826,2.232254,22749,40752.62,7,Elastic
2532,22423,-3.184956,14.607622,2505,30675.89,2,Elastic
406,47566,-2.464349,5.908978,5946,29499.25,8,Elastic
1090,20685,-2.576660,8.148875,2481,17614.00,6,Elastic
400,21754,-2.820667,6.630387,2782,16990.70,8,Elastic
827,22469,-2.055437,1.926008,9594,16549.12,7,Elastic
557,22383,-1.319090,2.014618,9347,15632.93,8,Elastic
1203,21232,-3.862724,1.508068,10865,13899.62,6,Elastic
1035,21928,-2.905887,2.323260,7398,13794.46,6,Elastic
275,84946,-3.585780,1.569496,10219,13627.77,9,Elastic


In [8]:
# ============================================================
# 7. INELASTIC PRODUCTS
# ============================================================

inelastic_products = product_elasticity[
    product_elasticity["Elasticity_Type"] == "Inelastic"
].copy()

inelastic_products = inelastic_products.sort_values(
    "Total_Revenue",
    ascending=False
)

print("\n========== HIGH-REVENUE INELASTIC PRODUCTS ==========")

display(
    inelastic_products[
        [
            "Stock_Code",
            "Average_Elasticity",
            "Average_Price",
            "Total_Quantity",
            "Total_Revenue",
            "Number_of_Months",
            "Elasticity_Type"
        ]
    ].head(10)
)



========== HIGH-REVENUE INELASTIC PRODUCTS ==========


,Stock_Code,Average_Elasticity,Average_Price,Total_Quantity,Total_Revenue,Number_of_Months,Elasticity_Type
10,M,-0.243670,436.843403,2623,207975.16,11,Inelastic
1,POST,-0.317668,26.362858,2185,46552.59,12,Inelastic
225,21929,-0.973992,2.336468,10996,20088.46,9,Inelastic
1250,21843,-0.182841,11.558097,1791,18657.63,6,Inelastic
1535,84879,-0.889354,2.260968,11268,18641.32,5,Inelastic
747,20914,-0.940293,3.190787,6593,18317.59,7,Inelastic
748,48194,-0.828854,8.350106,2147,14098.27,7,Inelastic
49,84406B,-0.086177,3.393945,4692,14086.78,11,Inelastic
556,22326,-0.280201,3.335724,4585,13411.79,8,Inelastic
1047,21915,-0.523203,1.383983,11724,13210.52,6,Inelastic


In [9]:

# ============================================================
# 8. TOP BUNDLE OPPORTUNITIES
# ============================================================

print("\n========== TOP BUNDLE OPPORTUNITIES ==========")

display(
    bundle_recommendations.head(20)
)


========== TOP BUNDLE OPPORTUNITIES ==========


,Product_A,Product_B,Support,Confidence_A_to_B,Lift,Combined_Revenue,Bundle_Score,Bundle_Recommendation,Business_Action
0,21231,21232,0.023195,0.485514,6.203040,51293.78,0.069855,Strong Bundle Candidate,Test Bundle Offer
1,20723,20724,0.014222,0.439528,8.923867,21876.91,0.055784,Strong Bundle Candidate,Cross-Selling Opportunity
2,20971,20972,0.012791,0.408537,10.503150,31090.64,0.054883,Strong Bundle Candidate,Cross-Selling Opportunity
3,84970L,84970S,0.012981,0.423017,9.761539,25355.04,0.053604,Strong Bundle Candidate,Cross-Selling Opportunity
4,21531,21535,0.010738,0.417440,11.374011,17492.79,0.050985,Strong Bundle Candidate,Cross-Selling Opportunity
5,20719,20724,0.014556,0.399738,8.115996,25290.31,0.047225,Strong Bundle Candidate,Cross-Selling Opportunity
6,22355,22356,0.015081,0.356659,8.679534,27156.19,0.046686,Strong Bundle Candidate,Cross-Selling Opportunity
7,84997C,84997D,0.010643,0.354531,11.716858,38968.61,0.044210,Strong Bundle Candidate,Cross-Selling Opportunity
8,22865,22866,0.010404,0.343849,11.753115,31565.90,0.042047,Strong Bundle Candidate,Cross-Selling Opportunity
9,84997B,84997C,0.010738,0.336323,11.203455,37349.42,0.040462,Strong Bundle Candidate,Cross-Selling Opportunity


In [10]:
# ============================================================
# 9. BUSINESS ACTION SUMMARY
# ============================================================

business_action_summary = (
    bundle_recommendations
    .groupby("Business_Action")
    .size()
    .reset_index(
        name="Bundle_Count"
    )
)

print("\n========== BUSINESS ACTION SUMMARY ==========")

display(
    business_action_summary
)


========== BUSINESS ACTION SUMMARY ==========


,Business_Action,Bundle_Count
0,Cross-Selling Opportunity,37
1,Monitor,1
2,Test Bundle Offer,36


In [11]:
# ============================================================
# 10. TOP BUNDLES BY LIFT
# ============================================================

top_lift_bundles = (
    bundle_recommendations
    .sort_values(
        "Lift",
        ascending=False
    )
)

print("\n========== TOP BUNDLES BY LIFT ==========")

display(
    top_lift_bundles[
        [
            "Product_A",
            "Product_B",
            "Support",
            "Confidence_A_to_B",
            "Lift",
            "Combined_Revenue",
            "Bundle_Score",
            "Bundle_Recommendation",
            "Business_Action"
        ]
    ].head(10)
)



========== TOP BUNDLES BY LIFT ==========


,Product_A,Product_B,Support,Confidence_A_to_B,Lift,Combined_Revenue,Bundle_Score,Bundle_Recommendation,Business_Action
8,22865,22866,0.010404,0.343849,11.753115,31565.90,0.042047,Strong Bundle Candidate,Cross-Selling Opportunity
7,84997C,84997D,0.010643,0.354531,11.716858,38968.61,0.044210,Strong Bundle Candidate,Cross-Selling Opportunity
4,21531,21535,0.010738,0.417440,11.374011,17492.79,0.050985,Strong Bundle Candidate,Cross-Selling Opportunity
9,84997B,84997C,0.010738,0.336323,11.203455,37349.42,0.040462,Strong Bundle Candidate,Cross-Selling Opportunity
2,20971,20972,0.012791,0.408537,10.503150,31090.64,0.054883,Strong Bundle Candidate,Cross-Selling Opportunity
3,84970L,84970S,0.012981,0.423017,9.761539,25355.04,0.053604,Strong Bundle Candidate,Cross-Selling Opportunity
1,20723,20724,0.014222,0.439528,8.923867,21876.91,0.055784,Strong Bundle Candidate,Cross-Selling Opportunity
6,22355,22356,0.015081,0.356659,8.679534,27156.19,0.046686,Strong Bundle Candidate,Cross-Selling Opportunity
15,22379,22381,0.010643,0.263282,8.606164,32281.50,0.024115,Strong Bundle Candidate,Cross-Selling Opportunity
5,20719,20724,0.014556,0.399738,8.115996,25290.31,0.047225,Strong Bundle Candidate,Cross-Selling Opportunity


In [12]:
# ============================================================
# 11. FINAL PRICING OPPORTUNITIES
# ============================================================

print("\n========== FINAL PRICING OPPORTUNITIES ==========")

display(
    pricing_opportunities.head(20)
)



========== FINAL PRICING OPPORTUNITIES ==========


,Stock_Code,Average_Elasticity,Average_Price,Total_Quantity,Total_Revenue,Number_of_Months,Elasticity_Type,Pricing_Strategy,Priority
0,M,-0.243670,436.843403,2623,207975.16,11,Inelastic,Test Moderate Price Increase,Medium
1,DOT,0.216521,172.396849,681,97834.13,12,Positive / Unusual,Review Further,Review
2,85123A,0.213845,3.020471,17335,46817.52,4,Positive / Unusual,Review Further,Review
3,POST,-0.317668,26.362858,2185,46552.59,12,Inelastic,Test Moderate Price Increase,Medium
4,85099B,-1.077826,2.232254,22749,40752.62,7,Elastic,Protect Price / Targeted Discount,High
5,47566,-2.464349,5.908978,5946,29499.25,8,Elastic,Protect Price / Targeted Discount,High
6,85099F,4.091470,2.227201,13135,23980.60,8,Positive / Unusual,Review Further,Review
7,20725,2.568798,2.075503,13470,22998.81,8,Positive / Unusual,Review Further,Review
8,21212,0.390219,0.672564,41557,22030.43,9,Positive / Unusual,Review Further,Review
9,21931,0.388710,2.327740,10727,20630.64,9,Positive / Unusual,Review Further,Review


In [13]:
# ============================================================
# 12. DATA QUALITY SUMMARY
# ============================================================

print("\n========== DATA QUALITY SUMMARY ==========")

display(
    data_quality_summary
)



========== DATA QUALITY SUMMARY ==========


,Metric,Value
0,Total Rows,525461
1,Total Columns,8
2,Duplicate Rows,6865
3,Cancellation Rows,10206
4,Negative Quantity Rows,12326
5,Zero Quantity Rows,0
6,Negative Price Rows,3
7,Zero Price Rows,3687
8,Missing Customer IDs,107927
9,Missing Dates,0


In [14]:
# ============================================================
# 13. CREATE FINAL INSIGHTS TABLE
# ============================================================

insights = []

insights.append([
    "Overall Business",
    "Total Revenue",
    total_revenue
])

insights.append([
    "Overall Business",
    "Total Units Sold",
    total_units
])

insights.append([
    "Overall Business",
    "Total Orders",
    total_orders
])

insights.append([
    "Overall Business",
    "Unique Products",
    total_products
])

insights.append([
    "Overall Business",
    "Unique Customers",
    total_customers
])

insights.append([
    "Overall Business",
    "Average Order Value",
    average_order_value
])

insights.append([
    "Overall Business",
    "Average Unit Price",
    average_unit_price
])

insights.append([
    "Pricing",
    "Elastic Products",
    len(
        product_elasticity[
            product_elasticity["Elasticity_Type"] == "Elastic"
        ]
    )
])

insights.append([
    "Pricing",
    "Inelastic Products",
    len(
        product_elasticity[
            product_elasticity["Elasticity_Type"] == "Inelastic"
        ]
    )
])

insights.append([
    "Bundling",
    "Bundle Candidates",
    len(bundle_recommendations)
])

insights_df = pd.DataFrame(
    insights,
    columns=[
        "Area",
        "Metric",
        "Value"
    ]
)

print("\n========== FINAL INSIGHTS TABLE ==========")

display(
    insights_df
)


========== FINAL INSIGHTS TABLE ==========


,Area,Metric,Value
0,Overall Business,Total Revenue,1.027214e+07
1,Overall Business,Total Units Sold,5.812948e+06
2,Overall Business,Total Orders,2.095300e+04
3,Overall Business,Unique Products,4.250000e+03
4,Overall Business,Unique Customers,4.312000e+03
5,Overall Business,Average Order Value,4.902466e+02
6,Overall Business,Average Unit Price,4.274692e+00
7,Pricing,Elastic Products,1.296000e+03
8,Pricing,Inelastic Products,6.150000e+02
9,Bundling,Bundle Candidates,7.400000e+01


In [15]:
# ============================================================
# 14. SAVE FINAL INSIGHTS
# ============================================================

insights_df.to_csv(
    "key_business_insights.csv",
    index=False
)

print(
    "\nkey_business_insights.csv saved successfully!"
)



key_business_insights.csv saved successfully!


In [16]:
# ============================================================
# 15. SAVE FINAL RECOMMENDATIONS
# ============================================================

recommendations = pd.DataFrame({

    "Area": [
        "Pricing",
        "Pricing",
        "Bundling",
        "Bundling",
        "Data Quality"
    ],

    "Recommendation": [
        "Monitor elastic products carefully before increasing prices because demand is more sensitive to price changes.",
        
        "Use controlled price testing for products classified as inelastic, especially where sufficient historical observations are available.",
        
        "Test bundles for product pairs showing strong lift and sufficient support.",
        
        "Use high-confidence product pairs as cross-selling opportunities.",
        
        "Continue monitoring missing customer IDs, duplicate transactions and invalid transaction records during future analysis."
    ]
})

display(
    recommendations
)

recommendations.to_csv(
    "business_recommendations.csv",
    index=False
)

print(
    "\nbusiness_recommendations.csv saved successfully!"
)

,Area,Recommendation
0,Pricing,Monitor elastic products carefully before incr...
1,Pricing,Use controlled price testing for products clas...
2,Bundling,Test bundles for product pairs showing strong ...
3,Bundling,Use high-confidence product pairs as cross-sel...
4,Data Quality,"Continue monitoring missing customer IDs, dupl..."



business_recommendations.csv saved successfully!
